In [1]:
] activate .

  Activating project at `~/devansh/kernel_tests_jl/SpatiotemporalGPs.jl/examples`


In [2]:
] st

Status `~/devansh/kernel_tests_jl/SpatiotemporalGPs.jl/examples/Project.toml`
  [052768ef] CUDA v5.8.3
  [91a5bcdd] Plots v1.40.17
  [73b3b457] SpatiotemporalGPs v1.0.1-DEV `..`
  [90137ffa] StaticArrays v1.9.14
  [37e2e46d] LinearAlgebra v1.11.0


In [3]:
using StaticArrays, LinearAlgebra, Plots

In [4]:
using SpatiotemporalGPs

In [10]:
using CUDA

In [213]:
CUDA.allowscalar(false)

In [92]:
# PREDICTION

N = 2000

x = randn(N)
sqrtP = randn(N, N)
P = sqrtP * sqrtP' + I
s = KFState(μ = x, Σ = P)

# dynamics (integrator)
A = [[zeros(N - 1);; I(N - 1)]; zeros(N)' ]
dt = 0.1
Ad = exp(A * dt) # convert to discrete time

# process noise
sqrtW = randn(N, N)
W = sqrtW * sqrtW' + I

# run the prediction
s_new = predict(s, Ad, W)

# test
P_new = Ad * P * Ad' + W
@assert get_μ(s_new) ≈ Ad * x
@assert Matrix(get_Σ(s_new)) ≈ P_new
@assert get_σ(s_new) ≈ sqrt.(diag(P_new))

In [93]:
cu_Ad = cu(Ad)
cu_W = cu(W)

2000×2000 CuArray{Float32, 2, CUDA.DeviceMemory}:
 2086.31      -34.3683      7.52918  …   -52.1772    -20.589       5.50405
  -34.3683   1901.58        2.50776       27.5546     36.5319   -121.013
    7.52918     2.50776  2084.8          -24.9908     27.8135      4.80853
  -14.173      89.2155    -34.4028         4.09446   -41.9967    -29.1783
   49.7332      4.45525    32.3991        27.0447    -39.5735     10.9267
   21.141      -6.33734   -62.9753   …   -60.0278     36.2865     23.0426
  -18.6475     -8.70819    83.202          2.61105    -8.97352   -55.8248
   23.3832     30.2151      7.75125      -77.7133     45.4139    -16.1058
    8.56066    53.7993    -97.9641       -12.0454    -33.0456     11.8201
   96.4461    -74.4212     84.8421      -140.866     -44.9739     63.2264
   55.7669     36.907      72.0425   …   -78.593      20.057      37.3046
  -15.9766     -3.6254     29.9792       -26.9436     26.8883     -7.18497
   14.015       3.39508   102.975         37.6167    -33.004

In [94]:
cu_s = KFState(μ = cu(x), Σ=cu(P))

KFState{Float32, CuArray{Float32, 1, CUDA.DeviceMemory}, CuArray{Float32, 2, CUDA.DeviceMemory}}(Float32[0.7607673, -0.37473425, -0.22030148, 0.6501609, -0.059927046, -0.25715062, 0.5726773, -1.2294395, -0.6407994, 0.69411564  …  -1.303271, 0.44722098, 0.40530306, -0.36949694, 1.3502496, 0.30447897, 0.9890992, 0.5694994, -1.5600749, 0.9025672], Float32[44.050087 -0.81025654 … -0.42445502 0.50706; 0.0 46.854286 … -0.9299517 -0.47178712; … ; 0.0 0.0 … 6.154907 -0.77384627; 0.0 0.0 … 0.0 7.4848113])

In [116]:
function LinearAlgebra.mul!(C::CuArray{F, 2, M},
        A::UpperTriangular{F, CuArray{F, 2, M}},
        B::Adjoint{F, CuArray{F, 2, M}}) where {F, M}
    
    # force the copy when running with adjoint
    return mul!(C, A, copy(B))
end

In [120]:
Ad

2000×2000 Matrix{Float64}:
 1.0  0.1  0.005  0.000166667  4.16667e-6   …  0.0          0.0
 0.0  1.0  0.1    0.005        0.000166667     0.0          0.0
 0.0  0.0  1.0    0.1          0.005           0.0          0.0
 0.0  0.0  0.0    1.0          0.1             0.0          0.0
 0.0  0.0  0.0    0.0          1.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0          …  0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0          …  0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 0.0  0.0  0.0    0.0          0.0             0.0          0.0
 ⋮                                          ⋱               
 0.0  0.0  0.0    0.0          0.0             2.75573e-17  2.51515e-19
 0.0  0.

In [125]:
s.U * Ad' - Array(cu_s.U * cu_Ad')

2000×2000 Matrix{Float64}:
 -4.01108e-6    3.0468e-8     2.81875e-9   …  -3.25528e-8   1.97626e-9
 -3.94875e-7   -4.37599e-6    3.29017e-7      -4.73096e-8  -4.82878e-8
  8.45592e-9   -1.71586e-7   -2.24346e-6       8.14345e-9  -7.3781e-9
  8.18399e-14  -1.21041e-8   -3.79883e-7      -1.94871e-8   4.30027e-9
 -2.38603e-11   6.86126e-11  -1.22452e-8       6.85821e-8   2.61664e-8
  9.72791e-14   3.10391e-12   1.02754e-10  …   6.60143e-9  -5.29671e-9
  2.16133e-15   3.4137e-14   -5.13321e-12      3.45271e-8   7.89488e-8
  1.12597e-17   4.76271e-16  -3.0728e-13       1.8272e-7   -1.62526e-9
  7.04232e-20   7.15234e-17   1.11182e-14      5.3286e-8   -1.02933e-7
  3.19572e-22  -9.53069e-20   1.23446e-16     -1.49819e-8   1.03289e-9
 -1.50732e-22  -1.95073e-20  -7.37011e-19  …   1.95969e-8   6.95371e-9
 -3.04913e-25   1.17967e-22   9.89833e-21      2.07357e-7  -1.8408e-8
 -9.78132e-27  -1.16305e-24   5.09278e-24     -3.11174e-8   1.46163e-7
  ⋮                                        ⋱        

In [121]:
s.U * Ad'

2000×2000 Matrix{Float64}:
 43.9694       -0.804618     …  -1.8606     -0.373749    0.50706
  4.69329      47.0139          -1.13893    -0.97713    -0.471787
  0.228157      4.56392          1.54791     0.267806    0.830712
  0.00768145    0.230371         0.551663   -1.84754     0.0946586
  0.000190535   0.00762133       0.178177    0.505496    0.683629
  3.7776e-6     0.000188866  …  -0.936341   -0.172302   -0.148329
  6.22873e-8    3.73677e-6      -0.615118   -0.113249    0.988164
  8.87218e-10   6.21076e-8      -1.1028     -2.01637    -0.0950044
  1.09597e-11   8.76748e-10      0.191241   -0.779269    0.856368
  1.22065e-13   1.09857e-11      0.609887    0.837244   -0.117233
  1.21338e-15   1.21339e-13  …   1.34766     0.484832    0.120008
  1.10103e-17   1.20636e-15      0.633573    2.53289     0.745401
  9.90254e-20   1.13878e-17      0.681617    0.0660949   2.65288
  ⋮                          ⋱                          
  0.0           0.0             -1.10474     0.0186257   0

In [127]:
CUDA.functional()

true

In [95]:
function cpu_predict(s::S, A, W) where {S <: KFState}
    N = length(s.μ)
    Γw = SpatiotemporalGPs.KalmanFilter.chol_sqrt(W)

    μ_new = A * s.μ
    F_new = SpatiotemporalGPs.KalmanFilter.qrr(s.U * A', Γw)

    return KFState(μ_new, F_new)
end

function gpu_predict(s::S, A, W) where {S <: KFState}
    N = length(s.μ)
    Γw = SpatiotemporalGPs.KalmanFilter.chol_sqrt(W)

    μ_new = A * s.μ
    F_new = SpatiotemporalGPs.KalmanFilter.qrr(s.U * copy(A'), Γw) # copy forces the transpose to be computed

    return KFState(μ_new, F_new)
end

gpu_predict (generic function with 1 method)

In [96]:
cpu_predict(s, Ad, W)

KFState{Float64, Vector{Float64}, Matrix{Float64}}([0.7223004505834603, -0.3935246026061397, -0.15562559976489074, 0.6429727326021715, -0.0829862428584563, -0.20613397432235928, 0.4466466491895085, -1.289985564739438, -0.5694870822100945, 0.7322131735093408  …  -1.2565783737430432, 0.4861301878682574, 0.3751595184571401, -0.23278247292074047, 1.3857315828411458, 0.405980149788793, 1.038399184159426, 0.41800473321025206, -1.4698181459094153, 0.9025672264413219], [-63.57434970243688 -2.3900451289108204 … 0.6537462729727389 -0.4054363217372071; 0.0 -64.24717067309281 … 0.10495538938400317 2.190789607527197; … ; 0.0 0.0 … -44.529992476133316 -1.9089202374239993; 0.0 0.0 … 0.0 -44.1241915051616])

In [97]:
gpu_predict(cu_s, cu_Ad, cu_W)

KFState{Float32, CuArray{Float32, 1, CUDA.DeviceMemory}, CuArray{Float32, 2, CUDA.DeviceMemory}}(Float32[0.7223004, -0.39352462, -0.1556256, 0.6429727, -0.08298624, -0.20613396, 0.44664666, -1.2899857, -0.5694871, 0.73221314  …  -1.2565784, 0.4861302, 0.37515953, -0.2327825, 1.3857315, 0.40598014, 1.0383992, 0.41800472, -1.4698182, 0.9025672], Float32[-63.574356 -2.3900452 … 0.65374625 -0.4054364; 0.0 -64.24718 … 0.104955316 2.1907895; … ; 0.0 0.0 … -44.529995 -1.9089196; 0.0 0.0 … 0.0 -44.124187])

In [126]:
cpu_predict(cu_s, cu_Ad, cu_W)

KFState{Float32, CuArray{Float32, 1, CUDA.DeviceMemory}, CuArray{Float32, 2, CUDA.DeviceMemory}}(Float32[0.7223004, -0.39352462, -0.1556256, 0.6429727, -0.08298624, -0.20613396, 0.44664666, -1.2899857, -0.5694871, 0.73221314  …  -1.2565784, 0.4861302, 0.37515953, -0.2327825, 1.3857315, 0.40598014, 1.0383992, 0.41800472, -1.4698182, 0.9025672], Float32[-63.574356 -2.3900452 … 0.65374625 -0.4054364; 0.0 -64.24718 … 0.104955316 2.1907895; … ; 0.0 0.0 … -44.529995 -1.9089196; 0.0 0.0 … 0.0 -44.124187])

In [98]:
@time cpu_predict(s, Ad, W);
@time gpu_predict(cu_s, cu_Ad, cu_W)

  0.555934 seconds (26 allocations: 153.107 MiB, 21.69% gc time)
  0.049398 seconds (686 allocations: 23.594 KiB)


KFState{Float32, CuArray{Float32, 1, CUDA.DeviceMemory}, CuArray{Float32, 2, CUDA.DeviceMemory}}(Float32[0.7223004, -0.39352462, -0.1556256, 0.6429727, -0.08298624, -0.20613396, 0.44664666, -1.2899857, -0.5694871, 0.73221314  …  -1.2565784, 0.4861302, 0.37515953, -0.2327825, 1.3857315, 0.40598014, 1.0383992, 0.41800472, -1.4698182, 0.9025672], Float32[-63.574356 -2.3900452 … 0.65374625 -0.4054364; 0.0 -64.24718 … 0.104955316 2.1907895; … ; 0.0 0.0 … -44.529995 -1.9089196; 0.0 0.0 … 0.0 -44.124187])

In [104]:
res = LinearAlgebra.LAPACK.geqrf!(cu_Ad)

(Float32[1.0 0.1 … 0.0 0.0; 0.0 1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.1; 0.0 0.0 … 0.0 1.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [106]:
@which LinearAlgebra.LAPACK.geqrf!(cu_Ad)

geqrf!(A::StridedCuMatrix{Float32})
     @ CUDA.CUSOLVER ~/.julia/packages/CUDA/Wfi8S/lib/cusolver/dense.jl:1027

In [110]:
@time LinearAlgebra.LAPACK.geqrf!(Ad)

  0.122271 seconds (8 allocations: 515.812 KiB)


([1.0 0.1 … 0.0 0.0; 0.0 1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.1; 0.0 0.0 … 0.0 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [111]:
@time LinearAlgebra.LAPACK.geqrf!(cu_Ad)

  0.014520 seconds (50 allocations: 1.203 KiB)


(Float32[1.0 0.1 … 0.0 0.0; 0.0 1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.1; 0.0 0.0 … 0.0 1.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [69]:
cu_Γw = SpatiotemporalGPs.KalmanFilter.chol_sqrt(cu_W)
blah = cu_s.U * copy(cu_Ad')
blah2 = SpatiotemporalGPs.KalmanFilter.qrr(blah, cu_Γw)
# SpatiotemporalGPs.KalmanFilter.qrr(blah, cu_Γw)

400×400 UpperTriangular{Float32, CuArray{Float32, 2, CUDA.DeviceMemory}}:
 -28.4465   -3.45956   -0.652481   -1.4248     …   -1.41766     -0.957981
    ⋅      -29.358     -1.37015    -0.0776581      -1.75754     -1.06353
    ⋅         ⋅       -28.6769     -2.2             1.28626      2.91681
    ⋅         ⋅          ⋅        -26.7087          0.98485      1.47749
    ⋅         ⋅          ⋅           ⋅             -0.253504    -0.968145
    ⋅         ⋅          ⋅           ⋅         …   -0.928715    -1.18538
    ⋅         ⋅          ⋅           ⋅             -0.067205     0.25622
    ⋅         ⋅          ⋅           ⋅             -0.507518     2.16967
    ⋅         ⋅          ⋅           ⋅             -1.44664     -0.416979
    ⋅         ⋅          ⋅           ⋅              0.426546    -0.734503
    ⋅         ⋅          ⋅           ⋅         …    0.485032     0.433648
    ⋅         ⋅          ⋅           ⋅              1.23284      0.268297
    ⋅         ⋅          ⋅           ⋅      

In [70]:
@time blah2 = SpatiotemporalGPs.KalmanFilter.qrr(blah, cu_Γw)

  0.003019 seconds (257 allocations: 7.312 KiB)


400×400 UpperTriangular{Float32, CuArray{Float32, 2, CUDA.DeviceMemory}}:
 -28.4465   -3.45956   -0.652481   -1.4248     …   -1.41766     -0.957981
    ⋅      -29.358     -1.37015    -0.0776581      -1.75754     -1.06353
    ⋅         ⋅       -28.6769     -2.2             1.28626      2.91681
    ⋅         ⋅          ⋅        -26.7087          0.98485      1.47749
    ⋅         ⋅          ⋅           ⋅             -0.253504    -0.968145
    ⋅         ⋅          ⋅           ⋅         …   -0.928715    -1.18538
    ⋅         ⋅          ⋅           ⋅             -0.067205     0.25622
    ⋅         ⋅          ⋅           ⋅             -0.507518     2.16967
    ⋅         ⋅          ⋅           ⋅             -1.44664     -0.416979
    ⋅         ⋅          ⋅           ⋅              0.426546    -0.734503
    ⋅         ⋅          ⋅           ⋅         …    0.485032     0.433648
    ⋅         ⋅          ⋅           ⋅              1.23284      0.268297
    ⋅         ⋅          ⋅           ⋅      

In [128]:
# run the prediction
cu_s_new = predict(cu_s, cu_Ad, cu_W)

KFState{Float32, CuArray{Float32, 1, CUDA.DeviceMemory}, CuArray{Float32, 2, CUDA.DeviceMemory}}(Float32[0.7223004, -0.39352462, -0.1556256, 0.6429727, -0.08298624, -0.20613396, 0.44664666, -1.2899857, -0.5694871, 0.73221314  …  -1.2565784, 0.4861302, 0.37515953, -0.2327825, 1.3857315, 0.40598014, 1.0383992, 0.41800472, -1.4698182, 0.9025672], Float32[-63.574356 -2.3900452 … 0.65374625 -0.4054364; 0.0 -64.24718 … 0.104955316 2.1907895; … ; 0.0 0.0 … -44.529995 -1.9089196; 0.0 0.0 … 0.0 -44.124187])

In [218]:
## CORRECTION
N = 2000 # number of states
M = 2 # number of measurements

x = randn(N)
sqrtP = randn(N, N)
P = sqrtP * sqrtP' + I
s = KFState(μ = x, Σ = P)

# measurement matrix
C = randn(M, N)

# measurement noise
sqrtV = randn(M, M)
V = sqrtV * sqrtV' + I

# create a measurement
y = C * x + randn(M)

# run the correction
@time s_new = correct(s, y, C, V)
@time s_new = correct(s, y, C, V)


# test
K = P * C' * inv(C * P * C' + V)
P_new = (I - K * C) * P
@assert get_μ(s_new) ≈ x + K * (y - C * x)
@assert Matrix(get_Σ(s_new)) ≈ P_new
@assert get_σ(s_new) ≈ sqrt.(diag(P_new))

  0.194858 seconds (69 allocations: 122.912 MiB, 1.35% gc time)
  0.170288 seconds (69 allocations: 122.912 MiB, 1.52% gc time)


In [219]:
cu_y = cu(y)
cu_C = cu(C)
cu_V = cu(V);

In [221]:
@time cu_s = KFState(μ=cu(x), Σ=cu(P))
@time cu_s_new = correct(cu_s, cu_y, cu_C, cu_V);

  0.009822 seconds (87 allocations: 15.268 MiB)
  0.021394 seconds (1.79 k allocations: 52.828 KiB)


In [223]:
CUDA.@profile correct(cu_s, cu_y, cu_C, cu_V);

In [224]:
0.171630/ 0.021

8.172857142857142

In [225]:
@assert Array(cu_s_new.μ) ≈ s_new.μ

In [226]:
@assert Array(cu_s_new.U) ≈ Array(s_new.U)

In [229]:
Γv = SpatiotemporalGPs.KalmanFilter.chol_sqrt(V)
@time L = SpatiotemporalGPs.KalmanFilter.kalman_gain(s, C, Γv)

  0.004345 seconds (28 allocations: 188.891 KiB)


2000×2 Matrix{Float64}:
 -0.00105975   -0.000152324
 -0.000975537  -6.76566e-5
  9.97867e-5   -0.000775277
  6.09883e-5    0.000255586
  6.03293e-5    0.000346151
  0.000106381   0.000628322
  0.000773994  -9.43852e-6
 -7.66492e-5   -0.00011326
 -0.000719201  -0.000791741
 -0.00104664   -0.00074154
 -0.00142005    0.000191627
  0.000297327  -0.000451129
  0.00010966   -0.000158431
  ⋮            
 -0.000118535   0.000271826
  0.000740012  -0.000346647
  1.56628e-5   -0.000421732
 -0.000310472   0.00110964
 -0.00103628   -0.000380726
 -0.000578972  -0.000930993
 -0.00152721   -0.000403254
 -0.00037572    0.000654503
  0.000130465  -0.000888194
  0.000532167  -5.7655e-5
  0.000389704   0.000509015
 -0.00102962   -0.000705708

In [230]:
cu_Γv = SpatiotemporalGPs.KalmanFilter.chol_sqrt(cu(V))
@time CUDA.@sync cu_L = SpatiotemporalGPs.KalmanFilter.kalman_gain(cu_s, cu_C, cu_Γv);

  0.000685 seconds (546 allocations: 14.641 KiB)


In [238]:
@time diag(s.U);

  0.000041 seconds (5 allocations: 15.734 KiB)


In [240]:
@time diag(cu_s.U);

  0.000169 seconds (67 allocations: 1.828 KiB)


In [244]:
@time Array(diag(cu_s.U))

  0.000246 seconds (73 allocations: 9.750 KiB)


2000-element Vector{Float32}:
 44.80356
 44.274902
 45.297863
 44.530224
 45.11142
 45.301605
 43.80436
 45.17369
 44.556824
 45.01669
 43.412277
 44.10099
 44.867603
  ⋮
  6.9623632
  6.983687
  7.260053
  7.9759383
  6.6886916
  6.9924436
  8.79238
  6.5129313
  7.421658
  6.83312
  7.452277
  7.142562